# Same matmul, five precisions
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week1/Numeric_Formats_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 1, Module 4: The programming stack**

Low precision is not only a smaller number of bits. The split between exponent bits and fraction bits sets the range and the precision of a format, and the hardware decides which formats run fast. FP16 has more fraction bits than BF16 but a much smaller range. FP8 comes in two encodings, E4M3 and E5M2, that trade range against precision.

In this notebook you compare FP32, FP16, BF16, FP8 and INT8 on three questions from the lecture: what values each format can represent, how fast the GPU multiplies matrices in it, and how much error it adds compared with an FP64 reference. Then you compute what each format means for the memory footprint of a model.

**What you will do**
1. Print the range and precision of each format.
2. Round a few values into each format and see where they break.
3. Time a 4096 by 4096 matrix multiply in each format the GPU supports.
4. Measure the error of each format against FP64.
5. Compute the weight footprint of Qwen 2.5 0.5B and of a 70B model in each format.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about two minutes.

## 1. Check the environment
The T4 is a Turing GPU. Its tensor cores run FP16 and INT8. It has no hardware support for BF16, TF32 or FP8, which arrived with later GPU generations. The timings in section 4 show what that means in practice.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {name}, compute capability {cc[0]}.{cc[1]}")
    print("Native BF16 support (compute capability 8.0 or later):", cc >= (8, 0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Range and precision of each format
Every floating-point format has a sign bit, some exponent bits and some fraction bits.

- **Exponent bits** set the range: the largest and smallest numbers you can store.
- **Fraction bits** set the precision. `eps` is the gap between 1.0 and the next representable number.

In [ ]:
formats = [
    ("FP32", torch.float32, 8, 23),
    ("FP16", torch.float16, 5, 10),
    ("BF16", torch.bfloat16, 8, 7),
    ("FP8 E4M3", torch.float8_e4m3fn, 4, 3),
    ("FP8 E5M2", torch.float8_e5m2, 5, 2),
]

print(f"{'format':9} {'bits':>4} {'exp':>4} {'frac':>4} {'eps':>11} {'max':>12} {'smallest normal':>16}")
for label, dt, e_bits, f_bits in formats:
    fi = torch.finfo(dt)
    print(f"{label:9} {fi.bits:4d} {e_bits:4d} {f_bits:4d} {fi.eps:11.3e} {fi.max:12.4g} {fi.tiny:16.3e}")

**Reading the output**
- BF16 has the same exponent width as FP32, so the same range (about 3.4e38). It keeps only 7 fraction bits, so `eps` is about 0.0078.
- FP16 has 10 fraction bits, so it is more precise than BF16, but its largest value is 65504.
- FP8 E4M3 has more precision and less range than E5M2. Its largest value is 448. E5M2 reaches 57344 with only two fraction bits.

## 3. Rounding values into each format
The cell stores a few values in each format and reads them back. A value outside the range becomes infinity or the largest finite value. A value with more digits than the format holds is rounded.

In [ ]:
values = torch.tensor([1 / 3, 1000.1, 70000.0, 1e-6, 3.14159], dtype=torch.float64)

print(f"{'value':>12}" + "".join(f"{label:>13}" for label, *_ in formats))
for v in values:
    row = []
    for _, dt, *_ in formats:
        back = v.to(dt).to(torch.float64).item()
        row.append(f"{back:13.6g}")
    print(f"{v.item():12.6g}" + "".join(row))

**Reading the output**
- 70000 overflows FP16 to `inf`. BF16 keeps it, with a rounding error.
- FP8 E4M3 has no infinity in this encoding. PyTorch saturates 1000.1 and 70000 to 448, its largest value. E5M2 rounds 70000 to `inf`. This is one reason FP8 is used with a scale factor per tensor.
- 1e-6 survives in FP32 and BF16. In FP16 and FP8 it becomes a subnormal or zero.
- 1000.1 shows the precision differences: BF16 and FP8 store a much coarser number than FP16.

## 4. Matmul speed in each format
The cell times `C = A @ B` for 4096 by 4096 matrices and reports the throughput. INT8 uses `torch._int_mm`, which multiplies INT8 matrices and accumulates in INT32. FP8 matrix multiply needs a newer GPU, so it is skipped here.

Throughput is `2 x 4096^3` operations divided by the measured time. The published T4 peaks are about 8 TFLOPS for FP32, 65 TFLOPS for FP16 on the tensor cores and 130 TOPS for INT8.

In [ ]:
import time


def time_op(fn, iters=20):
    for _ in range(3):
        fn()  # warm-up: the first call selects a kernel
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters


n = 4096
ops = 2 * n ** 3
results = {}
if device == "cuda":
    torch.manual_seed(0)
    a32 = torch.randn(n, n, device=device)
    b32 = torch.randn(n, n, device=device)
    for label, dt in (("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)):
        a, b = a32.to(dt), b32.to(dt)
        try:
            sec = time_op(lambda: a @ b)
            results[label] = ops / sec / 1e12
        except RuntimeError as err:
            results[label] = None
            print(f"{label}: not supported here ({str(err).splitlines()[0]})")
    a8 = torch.randint(-128, 128, (n, n), device=device, dtype=torch.int8)
    b8 = torch.randint(-128, 128, (n, n), device=device, dtype=torch.int8)
    try:
        sec = time_op(lambda: torch._int_mm(a8, b8))
        results["INT8"] = ops / sec / 1e12
    except RuntimeError as err:
        results["INT8"] = None
        print("INT8: not supported here", str(err).splitlines()[0])
    for label, tput in results.items():
        unit = "TOPS" if label == "INT8" else "TFLOPS"
        print(f"{label:5}: " + (f"{tput:6.1f} {unit}" if tput else "skipped"))
else:
    print("No GPU: skip this section.")

**Reading the output**
- FP16 is several times faster than FP32. The T4's tensor cores run FP16, and each value is half the size.
- BF16 runs, but slower than FP16 and even slower than FP32. The T4 has no BF16 tensor cores, so PyTorch falls back to slow kernels. On an A100 or H100, BF16 runs at the same speed as FP16.
- INT8 through `torch._int_mm` is slower than FP16 here, far below the published 130 TOPS. The T4 has INT8 tensor cores, but this PyTorch kernel does not drive them at full rate. A published peak says what the silicon can do, not what a given library delivers.
- None of these reach the published peak. Peaks assume ideal shapes and no memory stalls. Week 6 measures this gap.

## 5. Error against an FP64 reference
Speed is half of the trade. The cell multiplies two 1024 by 1024 matrices in each format and compares the result with the same product in FP64. The error is the relative Frobenius norm, `||C - C_ref|| / ||C_ref||`.

INT8 needs a scale factor, because the inputs are not integers. The cell uses one scale per matrix: divide by the largest absolute value, multiply by 127, round, and undo the scales after the integer matmul. FP8 values are converted back to FP32 before the multiply, so the FP8 rows show the cost of storing values in FP8, not of computing in it.

In [ ]:
torch.manual_seed(1)
m = 1024
a = torch.randn(m, m, dtype=torch.float64, device=device)
b = torch.randn(m, m, dtype=torch.float64, device=device)
ref = a @ b


def rel_err(c):
    return (torch.linalg.norm(c.double() - ref) / torch.linalg.norm(ref)).item()


errors = {}
for label, dt in (("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)):
    try:
        errors[label] = rel_err(a.to(dt) @ b.to(dt))
    except RuntimeError:
        errors[label] = None
for label, dt in (("FP8 E4M3 storage", torch.float8_e4m3fn), ("FP8 E5M2 storage", torch.float8_e5m2)):
    errors[label] = rel_err(a.to(dt).float() @ b.to(dt).float())

sa, sb = a.abs().max() / 127, b.abs().max() / 127
qa = torch.round(a / sa).to(torch.int8)
qb = torch.round(b / sb).to(torch.int8)
if device == "cuda":
    try:
        c8 = torch._int_mm(qa, qb).double() * sa * sb
        errors["INT8 (per-tensor scale)"] = rel_err(c8)
    except RuntimeError:
        errors["INT8 (per-tensor scale)"] = None

for label, err in errors.items():
    print(f"{label:24}: " + (f"{err:.2e}" if err is not None else "skipped"))

**Reading the output**
- FP32 is close to its rounding limit, about 1e-7.
- FP16 is about 1e-3 or below. cuBLAS accumulates FP16 products in FP32, which keeps the error down. Ask about the accumulation precision whenever a format is quoted.
- BF16 has about 8 times the error of FP16, matching its 3 fewer fraction bits.
- FP8 storage adds errors of a few percent. That is why FP8 inference uses scaling and keeps some operations in higher precision.
- INT8 with one scale per tensor lands between BF16 and FP8, at about 1.5 percent. INT8 has more levels than FP8 for values near the top of the range, but small values lose most of their digits. Real INT8 inference uses a scale per channel or per block.

## 6. Precision and memory footprint
Precision also decides whether a model fits and how many bytes each decode step reads. The lecture example is a 70-billion-parameter model. The cell repeats it and adds Qwen 2.5 0.5B, the model used in the labs.

In [ ]:
bytes_per_param = {"FP32": 4, "FP16 / BF16": 2, "FP8 / INT8": 1, "INT4": 0.5}
models = {"Qwen 2.5 0.5B": 494e6, "70B model": 70e9}

print(f"{'format':12}" + "".join(f"{name:>16}" for name in models))
for label, nbytes in bytes_per_param.items():
    print(f"{label:12}" + "".join(f"{params * nbytes / 1e9:13.1f} GB" for params in models.values()))

**Reading the output**
- The 70B model needs about 280 GB of weights at FP32, 140 at FP16, 70 at one byte and 35 at four bits, as on the lecture slide.
- These are only the weights. A deployment also needs the KV cache, activations, quantization scales and workspace.
- Fewer bytes per weight also means fewer bytes read per decode step. Week 4 shows that decode time is set mostly by those bytes.

## Try this
1. In section 4, change `n` to 256. The FP16 speedup over FP32 shrinks, because small matmuls are limited by overheads, not by the tensor cores.
2. In section 5, scale the inputs with `a = a * 1000`. FP16 now overflows. BF16 does not.
3. Replace the INT8 per-tensor scale with a scale per row of `a` and per column of `b`, and compare the error.

## Summary
- Exponent bits set the range, fraction bits set the precision. FP16 and BF16 make opposite choices.
- Speed depends on what the hardware supports: the T4 runs FP16 and INT8 fast, and BF16 slowly.
- Each format adds a known amount of error. Check the accumulation precision and the scaling, not only the storage format.
- Bytes per parameter set both the memory footprint and the bytes read in each decode step.

Lecture: Week 1, Module 4, slides 56 and 57 (numeric formats, precision and footprint).